# 04_1 — GTR Target Data Preparation

**Input** : `data/sentences_target_text_db.parquet`  
**Output** : `data/sentences_target_gtr_vector_db.index`

Encodes the target sentences with the **same GTR embedding pipeline that vec2text expects** (`T5 encoder + mean pooling`).  
This is intentionally **not** the full `SentenceTransformer` pipeline, because vec2text was trained on the embedder output before the extra SentenceTransformer projection/normalize stack.  
Run this once before `04_2_vec2text_inversion.ipynb`.

In [ ]:
from pathlib import Path

Path("data/sentences_target_gtr_vector_db.index").unlink(missing_ok=True)

In [ ]:
from gtr_runtime import load_gtr_encoder
import faiss
import pandas as pd

TARGET_TEXT_FILE  = "data/sentences_target_text_db.parquet"
TARGET_INDEX_FILE = "data/sentences_target_gtr_vector_db.index"
GTR_MODEL_NAME    = "sentence-transformers/gtr-t5-base"
PREFER_GPU        = True

In [ ]:
model, gtr_device = load_gtr_encoder(GTR_MODEL_NAME, prefer_gpu=PREFER_GPU)
print(f"Loaded {GTR_MODEL_NAME} on {gtr_device}")

probe_emb = model.encode(
    ["hello world"],
    batch_size=1,
    convert_to_numpy=True,
    normalize_embeddings=False,
    show_progress_bar=False,
)
print(f"Embedding shape: {probe_emb.shape} | L2 norm: {(probe_emb[0] ** 2).sum() ** 0.5:.4f}")

In [ ]:
target_df  = pd.read_parquet(TARGET_TEXT_FILE)
target_emb = model.encode(
    target_df["text"].tolist(),
    batch_size=8,
    convert_to_numpy=True,
    normalize_embeddings=False,
    show_progress_bar=False,
)

In [ ]:
target_index = faiss.IndexFlatIP(target_emb.shape[1])
target_index.add(target_emb)
faiss.write_index(target_index, TARGET_INDEX_FILE)
print(f"Saved {target_index.ntotal} GTR vectors → {TARGET_INDEX_FILE}")